In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 00. LLM数学学習の始め方

> **学習目標**
> - LLM
> - 内容 学習
> - Colab CPU/GPU
> - 共通ユーティリティ `llm_math`

## 0.1 "" LLM?

ChatGPT, GPT-4, LLaMA, Claude... LLM( モデル) 2022 .
 API , モデル .

 ** GPT ** .
- 行列積 $AB$ Attention ?
- $\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$ 分布 ?
- バックプロパゲーション $\frac{d}{dx}f(g(x)) = f'(g(x)) g'(x)$ ?

 , LLM "" "" "" .

## 0.2 内容

```
 →  → 複雑度 → → ベンチマーク →
```

 6 内容.

## 0.3

Colab . .


In [ ]:
#  
import sys
print(f"Python: {sys.version.split()[0]}")

#  
import numpy as np
print(f"NumPy: {np.__version__}")

import matplotlib
print(f"Matplotlib: {matplotlib.__version__}")

import scipy
print(f"SciPy: {scipy.__version__}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch: NOT installed (Part 2   )")


### llm_math パッケージの読み込み

 共通ユーティリティ `llm_math` .
 . ( .)


In [ ]:
# preamble  llm_math  
if _LLM_MATH_OK:
    print("✓ llm_math パッケージの読み込みエラー")
    print(f"  - viz: Visualization  ({viz.__name__})")
    print(f"  - bench: CPU vs GPU ベンチマーク  ({bench.__name__})")
    print(f"  - data:  データ  ({data.__name__})")
else:
    print("⚠  .    :")
    print("  !git clone https://bit.ly/4f1YusE")
    print("  %cd llm-math-book")
    print("  !bash colab_setup.sh")


## 0.4 ベンチマーク — 行列積 CPU/GPU 比較

 **CPU GPU 速度 ** .
 行列積 .


In [ ]:
#  ベンチマーク: 行列積 CPU vs GPU
import time
import numpy as np

def bench_np_matmul(n, repeat=5):
    """NumPy Matrix Time Measurement (CPU)."""
    A = np.random.randn(n, n).astype(np.float32)
    B = np.random.randn(n, n).astype(np.float32)
    # warmup
    for _ in range(2):
        _ = A @ B
    # Measurement
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        _ = A @ B
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)
    return np.mean(times), np.std(times)

# Matrix Magnitude Measurement
for n in [64, 256, 1024, 2048]:
    mean_ms, std_ms = bench_np_matmul(n, repeat=3)
    print(f"  n={n:5d}: {mean_ms:8.3f} ± {std_ms:6.3f} ms")


In [ ]:
# PyTorch  CPU vs GPU 比較
try:
    import torch
    print("PyTorch CPU vs GPU Matrix Comparison:\n")
    print(f"{'n':>6} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
    print("-" * 50)
    for n in [64, 256, 1024, 2048]:
        # CPU
        A_cpu = torch.randn(n, n)
        B_cpu = torch.randn(n, n)
        for _ in range(2): _ = A_cpu @ B_cpu  # warmup
        t0 = time.perf_counter()
        for _ in range(3):
            _ = A_cpu @ B_cpu
        cpu_ms = (time.perf_counter() - t0) / 3 * 1000

        if torch.cuda.is_available():
            A_gpu = A_cpu.cuda()
            B_gpu = B_cpu.cuda()
            for _ in range(2): _ = A_gpu @ B_gpu  # warmup
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(3):
                _ = A_gpu @ B_gpu
            torch.cuda.synchronize()
            gpu_ms = (time.perf_counter() - t0) / 3 * 1000
            speedup = cpu_ms / gpu_ms
            print(f"{n:>6} | {cpu_ms:>12.3f} | {gpu_ms:>12.3f} | {speedup:>9.2f}x")
        else:
            print(f"{n:>6} | {cpu_ms:>12.3f} | {'N/A':>12} | {'N/A':>10}")
    if not torch.cuda.is_available():
        print("\n⚠ GPU . Colab  T4 GPU : Runtime → Change runtime type")
except ImportError:
    print("PyTorch    .")


## 0.5 ベンチマーク

 ベンチマーク :

1. **Warm-up**: 2~3 /
2. ** **: 5~10 ·
3. ****: GPU `torch.cuda.synchronize`
4. ** **: GPU `torch.cuda.max_memory_allocated`
5. ** データ**: CPU/GPU

 `llm_math.bench` .


In [ ]:
# llm_math.bench   
import torch
from llm_math.bench import time_fn, format_results_table, get_device

device = get_device()
print(f" : {device}")

def matmul_op(A, B):
    return A @ B

results = {}
for dev_name in ['cpu'] + (['cuda'] if torch.cuda.is_available() else []):
    n = 1024
    A = torch.randn(n, n, device=dev_name)
    B = torch.randn(n, n, device=dev_name)
    res = time_fn(matmul_op, A, B, device=dev_name, warmup=2, repeat=5)
    results[dev_name.upper()] = res

print(format_results_table(results, title="Matrix (n=1024) ベンチマーク"))


## 0.6 演習問題 解答

 演習問題 . 解答 `solutions/chXX_solutions.ipynb` .

## 0.7

**Ch 01. ベクトル 空間** LLM ベクトル .
, Attention, — ベクトル 空間 .

---

###
- LLM (··)
- : → → Attention → → LLM
- Colab CPU/GPU
- 共通ユーティリティ `llm_math`
